# Human Face Detection System


Build a computer vision system that takes a single RGB image as input and determines whether a human face is present in the image. If a face is detected, the system should draw a green bounding box around the detected face(s) and display the message "Human face detected".

First, implement this system using classical computer vision techniques, specifically the Haar Cascade classifier.

Then, extend the same task using a deep learning approach, employing a Deep Neural Network (DNN) for face detection.

Finally, compare the performance of both approaches in terms of accuracy, speed, robustness, and limitations.


## Upload File(s) from the Local Machine


In [ ]:
import os
from google.colab import files

# Select a file from the local machine
uploaded = files.upload()

for fn in uploaded.keys():
    print(f"User uploaded file '{fn}' with length {len(uploaded[fn])} bytes")

## Face Detection using Haar Cascade Classifier


In [ ]:
import cv2
from google.colab.patches import cv2_imshow

# List of common image extensions
image_extensions = (".jpg", ".jpeg", ".png", ".bmp")

# Get all image files in the current directory
all_files = os.listdir(".")
image_files = [f for f in all_files if f.lower().endswith(image_extensions)]

print(f"All files in directory: {all_files}")

if not image_files:
    print("No image files found. Please upload images to the Colab file pane.")
else:
    # Load the cascade classifier
    cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    face_cascade = cv2.CascadeClassifier(cascade_path)

    if face_cascade.empty():
        print("Error: Could not load cascade classifier")
    else:
        for image_path in image_files:
            print(f"\nProcessing: {image_path}")
            image = cv2.imread(image_path)

            if image is None:
                print(f"Error: Could not decode image {image_path}.")
                continue

            # Convert to grayscale
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

            # Detect faces
            faces = face_cascade.detectMultiScale(
                gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
            )

            # Draw results
            if len(faces) > 0:
                print(f"{len(faces)} human face(s) detected in {image_path}")
                for x, y, w, h in faces:
                    cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
            else:
                print(f"No human detected in {image_path}")

            # Show result
            cv2_imshow(image)

## Face Detection using DNN Classifier


In [ ]:
import numpy as np
import urllib.request

# Model file paths
prototxt = "deploy.prototxt"
model = "res10_300x300_ssd_iter_140000.caffemodel"

# Download model files if they don't exist
if not os.path.exists(prototxt):
    url_proto = "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt"
    urllib.request.urlretrieve(url_proto, prototxt)
if not os.path.exists(model):
    url_model = "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel"
    urllib.request.urlretrieve(url_model, model)

# Load DNN model
net = cv2.dnn.readNetFromCaffe(prototxt, model)

# Define supported image extensions
image_extensions = (".jpg", ".jpeg", ".png", ".bmp")
image_files = [f for f in os.listdir(".") if f.lower().endswith(image_extensions)]

if not image_files:
    print("No image files found in the directory.")
else:
    for image_path in image_files:
        print(f"\n--- Processing: {image_path} ---")
        image = cv2.imread(image_path)
        if image is None:
            print(f"Error: Could not read {image_path}.")
            continue

        (h, w) = image.shape[:2]

        # Preprocess image (blob)
        blob = cv2.dnn.blobFromImage(
            cv2.resize(image, (300, 300)), 1.0, (300, 300), (104.0, 177.0, 123.0)
        )

        net.setInput(blob)
        detections = net.forward()

        # Process detections
        face_count = 0
        for i in range(detections.shape[2]):
            confidence = detections[0, 0, i, 2]
            if confidence > 0.5:
                face_count += 1
                box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                (x1, y1, x2, y2) = box.astype("int")
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

        if face_count > 0:
            print(f"Human detected: {face_count} face(s) found.")
        else:
            print("No human detected")

        # Show result using Colab patch
        cv2_imshow(image)

In [ ]:
import glob

# Define supported image extensions
image_extensions = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.prototxt", "*.caffemodel")

files_removed = 0
for ext in image_extensions:
    for f in glob.glob(ext):
        try:
            os.remove(f)
            print(f"Removed: {f}")
            files_removed += 1
        except OSError as e:
            print(f"Error: {f} : {e.strerror}")

if files_removed == 0:
    print("No image files were found to remove.")
else:
    print(f"Total files removed: {files_removed}")